# Command Execution Notebook

This notebook runs four requested commands in `c:/Users/user01/Desktop/lab_control` and reports concise outputs with exit codes, including failure details.

In [4]:
from __future__ import annotations

import os
import subprocess
from dataclasses import dataclass
from typing import List

WORKSPACE = r"c:/Users/user01/Desktop/lab_control"

COMMANDS = [
    [r"c:/Users/user01/Desktop/lab_control/.venv/Scripts/python.exe", "-m", "pytest", "tests/test_flipper.py", "tests/test_gui_smoke.py", "-q"],
    [r"c:/Users/user01/Desktop/lab_control/.venv/Scripts/python.exe", "main.py", "discover", "--config", "config/instruments.yaml"],
    [r"c:/Users/user01/Desktop/lab_control/.venv/Scripts/python.exe", "tests/manual/test_flipper_live.py"],
    [r"c:/Users/user01/Desktop/lab_control/.venv/Scripts/python.exe", "tests/manual/test_kdc101_live.py"],
]

@dataclass
class CommandResult:
    step: int
    command: str
    exit_code: int
    stdout: str
    stderr: str
    concise_output: str
    failure_details: str


def _tail(text: str, n: int = 20) -> str:
    lines = [line for line in text.splitlines() if line.strip()]
    if not lines:
        return ""
    return "\n".join(lines[-n:])


def _decode_stream(data: bytes | None) -> str:
    if not data:
        return ""
    for enc in ("utf-8", "cp1252", "cp950"):
        try:
            return data.decode(enc)
        except Exception:
            continue
    return data.decode("utf-8", errors="replace")


def run_command(step: int, cmd: List[str], timeout: int = 300) -> CommandResult:
    cmd_str = " ".join(cmd)
    try:
        completed = subprocess.run(
            cmd,
            cwd=WORKSPACE,
            capture_output=True,
            text=False,
            timeout=timeout,
            env=os.environ.copy(),
        )
        stdout = _decode_stream(completed.stdout)
        stderr = _decode_stream(completed.stderr)
        merged = (stdout + "\n" + stderr).strip()
        concise = _tail(merged, 20)
        failure = ""
        if completed.returncode != 0:
            failure = _tail(stderr, 40) or _tail(stdout, 40) or "No stderr/stdout captured."
        return CommandResult(
            step=step,
            command=cmd_str,
            exit_code=completed.returncode,
            stdout=stdout,
            stderr=stderr,
            concise_output=concise,
            failure_details=failure,
        )
    except subprocess.TimeoutExpired as exc:
        out = _decode_stream(exc.stdout if isinstance(exc.stdout, (bytes, bytearray)) else None)
        err = _decode_stream(exc.stderr if isinstance(exc.stderr, (bytes, bytearray)) else None)
        merged = (out + "\n" + err + "\nTimed out.").strip()
        return CommandResult(
            step=step,
            command=cmd_str,
            exit_code=124,
            stdout=out,
            stderr=err,
            concise_output=_tail(merged, 20),
            failure_details=f"Timeout after {timeout}s\n" + (_tail(err, 40) or _tail(out, 40)),
        )
    except Exception as exc:
        msg = f"Exception: {type(exc).__name__}: {exc}"
        return CommandResult(
            step=step,
            command=cmd_str,
            exit_code=1,
            stdout="",
            stderr=msg,
            concise_output=msg,
            failure_details=msg,
        )

print(f"Workspace: {WORKSPACE}")
for idx, c in enumerate(COMMANDS, start=1):
    print(f"{idx}) {' '.join(c)}")

Workspace: c:/Users/user01/Desktop/lab_control
1) c:/Users/user01/Desktop/lab_control/.venv/Scripts/python.exe -m pytest tests/test_flipper.py tests/test_gui_smoke.py -q
2) c:/Users/user01/Desktop/lab_control/.venv/Scripts/python.exe main.py discover --config config/instruments.yaml
3) c:/Users/user01/Desktop/lab_control/.venv/Scripts/python.exe tests/manual/test_flipper_live.py
4) c:/Users/user01/Desktop/lab_control/.venv/Scripts/python.exe tests/manual/test_kdc101_live.py


## Run Command 1: Pytest (test_flipper.py, test_gui_smoke.py)

In [5]:
results = []

r1 = run_command(1, COMMANDS[0])
results.append(r1)
print(f"[1] Exit Code: {r1.exit_code}")
print(r1.concise_output)
if r1.exit_code != 0:
    print("\n[1] FAILURE DETAILS:")
    print(r1.failure_details)

[1] Exit Code: 0
============================= test session starts =============================
platform win32 -- Python 3.12.10, pytest-9.0.2, pluggy-1.6.0
rootdir: c:\Users\user01\Desktop\lab_control
configfile: pyproject.toml
plugins: mock-3.15.1
collected 13 items
tests\test_flipper.py .....                                              [ 38%]
tests\test_gui_smoke.py ........                                         [100%]
============================= 13 passed in 1.61s ==============================


## Run Command 2: main.py discover with config

In [6]:
r2 = run_command(2, COMMANDS[1])
results.append(r2)
print(f"[2] Exit Code: {r2.exit_code}")
print(r2.concise_output)
if r2.exit_code != 0:
    print("\n[2] FAILURE DETAILS:")
    print(r2.failure_details)

[2] Exit Code: 0
╔══════════════════════════════════╗
║   Lab Instrument Control v0.1   ║
║   2026-03-06 19:41:23   ║
╚══════════════════════════════════╝
VISA Resources:
  USB0::0x1313::0x8078::P0023709::INSTR              -> Thorlabs,PM100D,P0023709,2.6.0
Kinesis Devices:
  37007871 -> APT Filter Flipper
  27005601 -> Brushed Motor Controller


## Run Command 3: Manual Flipper Live Test

In [7]:
r3 = run_command(3, COMMANDS[2])
results.append(r3)
print(f"[3] Exit Code: {r3.exit_code}")
print(r3.concise_output)
if r3.exit_code != 0:
    print("\n[3] FAILURE DETAILS:")
    print(r3.failure_details)

[3] Exit Code: 0
MFF:37007871
Initial state: 1
Toggled to: 2
Toggled to: 2
Flipper live test complete
c:\Users\user01\Desktop\lab_control\.venv\Lib\site-packages\pylablib\devices\Thorlabs\kinesis.py:230: UserWarning: model number MFF002 doesn't match the device ID prefix 37(MFF10.)
  warnings.warn("model number {} doesn't match the device ID prefix {}({})".format(model_no,port,port_model_no))


## Run Command 4: Manual KDC101 Live Test

In [8]:
r4 = run_command(4, COMMANDS[3])
results.append(r4)
print(f"[4] Exit Code: {r4.exit_code}")
print(r4.concise_output)
if r4.exit_code != 0:
    print("\n[4] FAILURE DETAILS:")
    print(r4.failure_details)

[4] Exit Code: 0
KDC101:27005601
Angle: 0.000 deg
Angle: 45.001 deg
Angle: 90.001 deg
Angle: 180.001 deg
Angle: -0.001 deg
All moves complete


## Build Concise Execution Report with Failure Details

In [9]:
rows = []
for r in results:
    rows.append({
        "step": r.step,
        "command": r.command,
        "exit_code": r.exit_code,
        "status": "PASS" if r.exit_code == 0 else "FAIL",
        "concise_output": r.concise_output,
    })

try:
    import pandas as pd
    display(pd.DataFrame(rows, columns=["step", "command", "exit_code", "status", "concise_output"]))
except Exception:
    for row in rows:
        print(row)

for r in results:
    if r.exit_code != 0:
        print(f"\n--- FAILURE STEP {r.step} ---")
        print(r.failure_details)

,step,command,exit_code,status,concise_output
0,1,c:/Users/user01/Desktop/lab_control/.venv/Scri...,0,PASS,[1m============================= test session...
1,2,c:/Users/user01/Desktop/lab_control/.venv/Scri...,0,PASS,╔══════════════════════════════════╗\n║ Lab ...
2,3,c:/Users/user01/Desktop/lab_control/.venv/Scri...,0,PASS,MFF:37007871\nInitial state: 1\nToggled to: 2\...
3,4,c:/Users/user01/Desktop/lab_control/.venv/Scri...,0,PASS,KDC101:27005601\nAngle: 0.000 deg\nAngle: 45.0...
